# Supplemental Tables for PepBind3D Manuscript

Generates six supplementary tables:

| Table | Content | Source |
|-------|---------|--------|
| **S1** | Curation pipeline decision logic | Hand-encoded from `clean_peplist()` in `IEDBTestPipeline.py` |
| **S2** | Record attrition funnel | `analysis/attrition_counts.py` |
| **S3** | Per-allele Spearman correlations (IC50 and Kd) | Computed from `metadata.csv` |
| **S4** | Per-pair structural validation | `01_structural_regen/rmsd_per_pair.csv` |
| **S5** | Per-allele dataset composition | Derived from `metadata.csv` |
| **S6** | Score-metric comparison | `scores_out/metric_comparison_pooled.csv` |

The last cell writes one CSV per table plus a combined `.xlsx`, and copies both
into `revisions/tables/` for the draft.

## Imports and paths

In [1]:
from paths import DATA_ROOT, MHC_DB_ROOT, CLUSTER_ROOT  # roots; override with PEPBIND3D_* env vars
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from pathlib import Path
import openpyxl

HF_DIR        = DATA_ROOT / "huggingface"
METADATA_CSV  = HF_DIR / 'metadata.csv'
SCORES_DIR    = DATA_ROOT / "IEDB_validation/scores_out"
OUT_01        = DATA_ROOT / "IEDB_validation/01_structural_regen"
RMSD_CSV      = OUT_01 / 'rmsd_per_pair.csv'
OUT_DIR       = DATA_ROOT / "IEDB_validation/supplemental_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

## Table S1: Curation pipeline decision logic

In [2]:
s1_rows = [
    {'Step': 'Pre-filter', 'Case': 'Missing quantitative measurement or assay units',
     'Condition': '`Quantitative_Measurement` or `Assay_Units` is NaN',
     'Action': 'Drop record'},
    {'Step': 'Pre-filter', 'Case': 'Variant Kd assay labels',
     'Condition': '`Assay_Response_Measured` ∈ {dissociation constant KD (~EC50), dissociation constant KD, dissociation constant (~IC50)}',
     'Action': 'Normalize to dissociation constant (KD) prior to filtering'},
    {'Step': 'Pre-filter', 'Case': 'Non-binding-affinity assay',
     'Condition': '`Assay_Response_Standardized` not in {dissociation constant (KD), half maximal inhibitory concentration (IC50)}',
     'Action': 'Drop record'},

    {'Step': 'Dedup (n=1)', 'Case': 'Single record',
     'Condition': ', ', 'Action': 'Keep as-is'},

    {'Step': 'Dedup (n=2)', 'Case': 'PubMed provenance asymmetric',
     'Condition': 'One record has a PubMed ID, the other does not',
     'Action': 'Drop the record without PubMed ID'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both lack PubMed, values agree',
     'Condition': 'Both records lack PubMed ID AND |Δvalue| ≤ 10 nM',
     'Action': 'Keep one representative (the first)'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both lack PubMed, values disagree',
     'Condition': 'Both records lack PubMed ID AND |Δvalue| > 10 nM',
     'Action': 'Drop both records'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both have PubMed, values agree',
     'Condition': 'Both records have PubMed ID AND |Δvalue| < 10 nM',
     'Action': 'Keep one representative'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both have PubMed, values disagree',
     'Condition': 'Both records have PubMed ID AND |Δvalue| ≥ 10 nM',
     'Action': 'Flag for manual review against source publications'},

    {'Step': 'Dedup (n>2)', 'Case': 'Mixed PubMed status',
     'Condition': 'Some records have PubMed ID, others do not',
     'Action': 'Drop records without PubMed ID first, then re-evaluate'},
    {'Step': 'Dedup (n>2)', 'Case': 'All values within 10 nM',
     'Condition': 'All pairwise |Δvalue| ≤ 10 nM after PubMed filter',
     'Action': 'Keep one representative'},
    {'Step': 'Dedup (n>2)', 'Case': 'Values disagree by >10 nM',
     'Condition': 'At least one pair of records differs by >10 nM',
     'Action': 'Keep the value closest to the median'},

    {'Step': 'Post-filter', 'Case': 'Non-canonical amino acids',
     'Condition': 'Peptide sequence contains the "+" character',
     'Action': 'Exclude from structure generation'},
]

S1 = pd.DataFrame(s1_rows)
S1

,Step,Case,Condition,Action
0,Pre-filter,Missing quantitative measurement or assay units,`Quantitative_Measurement` or `Assay_Units` is NaN,Drop record
1,Pre-filter,Variant Kd assay labels,"`Assay_Response_Measured` ∈ {dissociation constant KD (~EC50), dissociation constant KD, dissociation constant (~IC50)}",Normalize to dissociation constant (KD) prior to filtering
2,Pre-filter,Non-binding-affinity assay,"`Assay_Response_Standardized` not in {dissociation constant (KD), half maximal inhibitory concentration (IC50)}",Drop record
3,Dedup (n=1),Single record,",",Keep as-is
4,Dedup (n=2),PubMed provenance asymmetric,"One record has a PubMed ID, the other does not",Drop the record without PubMed ID
5,Dedup (n=2),"Both lack PubMed, values agree",Both records lack PubMed ID AND |Δvalue| ≤ 10 nM,Keep one representative (the first)
6,Dedup (n=2),"Both lack PubMed, values disagree",Both records lack PubMed ID AND |Δvalue| > 10 nM,Drop both records
7,Dedup (n=2),"Both have PubMed, values agree",Both records have PubMed ID AND |Δvalue| < 10 nM,Keep one representative
8,Dedup (n=2),"Both have PubMed, values disagree",Both records have PubMed ID AND |Δvalue| ≥ 10 nM,Flag for manual review against source publications
9,Dedup (n>2),Mixed PubMed status,"Some records have PubMed ID, others do not","Drop records without PubMed ID first, then re-evaluate"


## Table S2: Record attrition funnel

In [3]:
# Record attrition at each IEDB filtering stage (Rocco comment 5).
# Generated by analysis/attrition_counts.py; adjust the path if it wrote elsewhere.
ATTRITION_CSV = SCORES_DIR.parent / 'supplementary_table_S1_attrition.csv'
S2 = pd.read_csv(ATTRITION_CSV)
S2 = S2.rename(columns={'Note': 'Filter applied', 'Removed': 'Records removed'})
S2 = S2[['Stage', 'Filter applied', 'Records remaining', 'Records removed']]
S2

,Stage,Filter applied,Records remaining,Records removed
0,0. Raw IEDB MHC ligand records,IEDB MHC ligand bulk download,4883585,NaN
1,1. HLA-A / HLA-B / HLA-C allele restriction,"MHC allele name begins with HLA-A, HLA-B or HLA-C",1532863,3350722.0
2,2. Quantitative value + assay units present,Both a quantitative measurement value and assay units,147967,1384896.0
3,3. Retained assay response (KD or IC50),Assay response is KD or IC50 (variant KD labels normalized),134840,13127.0
4,"4. Deduplication, flagging, and sequence filtering","Duplicate resolution, flagged-record removal, non-canonical residue and length exclusion",119316,15524.0
5,5. Pairs with a generated structural ensemble,Curated pairs for which structure generation completed,118985,331.0


## Table S3: Per-allele Spearman correlations

In [4]:
df = pd.read_csv(METADATA_CSV)
print('Rows:', len(df))
df = df[~df['flagged'].astype(bool)].copy()
print('After dropping flagged:', len(df))

print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nUnique measurement_type values:', df['measurement_type'].unique())

Rows: 118985
After dropping flagged: 118970
Shape: (118970, 28)
Columns: ['allele_iedb', 'allele', 'allele_compact', 'peptide', 'peptide_length', 'measurement_type', 'measurement_value', 'measurement_units', 'assay_method', 'assay_response', 'pubmed_id', 'parent_protein', 'protein_accession', 'source_organism', 'source_version', 'flagged', 'self_templated', 'has_structures', 'num_pdbs', 'I_sc_best', 'I_sc_mean', 'reweighted_sc_best', 'reweighted_sc_mean', 'total_score_best', 'total_score_mean', 'pdb_dir', 'pep_sc_best', 'pep_sc_mean']

Unique measurement_type values: ['IC50' 'Kd']


In [5]:
# Robust filter: case-insensitive substring match
type_lower = df['measurement_type'].astype(str).str.lower()
is_ic50 = type_lower.str.contains('ic50', na=False)
is_kd   = type_lower.str.contains(r'\bkd\b', na=False, regex=True) | type_lower.str.fullmatch('kd', na=False)

print(f'IC50 rows: {is_ic50.sum():,}    Kd rows: {is_kd.sum():,}')

ic50 = df[is_ic50].copy()
kd   = df[is_kd].copy()

IC50 rows: 21,227    Kd rows: 97,743


In [6]:
MIN_N = 10
IC50_CENSORED, IC50_FLOOR = {20000, 50000, 70000}, 70000
KD_CENSORED,   KD_FLOOR   = {5000, 10000, 20000},  20000
SCORE_COL = 'I_sc_best'

# Scores come from metadata.csv itself (release/add_release_columns.py).
# Do NOT merge scores_out/score_summary.csv: it covers only the 49,268 v1
# pairs and predates the re-dock of the 1,162 content-defective ones, so it
# would cut the dataset by more than half and restore their old scores
# (median I_sc_best -11.60 REU against -68.57 after the re-dock).
# This cell used to drop metadata's score columns and merge that file, which
# raised no error: it just silently computed the table on the stale half.
assert SCORE_COL in df.columns, (f'metadata.csv lacks {SCORE_COL}; regenerate '
                                 'it with release/add_release_columns.py')
df['allele_dir'] = df['allele'].map(
    lambda a: (a[4:] if a.startswith('HLA-') else a).replace('*','').replace(':',''))
print(f'rows without {SCORE_COL}: {df[SCORE_COL].isna().sum():,}')

# Rebuild the subsets after the merge so they carry the score columns
type_lower = df['measurement_type'].astype(str).str.lower()
is_ic50 = type_lower.str.contains('ic50', na=False)
is_kd   = type_lower.str.fullmatch('kd', na=False)
ic50 = df[is_ic50].copy()
kd   = df[is_kd].copy()
print(f'IC50 rows: {len(ic50):,}   Kd rows: {len(kd):,}')

def per_allele_spearman(df_subset, censored, floor, measurement_label):
    keep = ~(df_subset['measurement_value'].isin(censored)
             | (df_subset['measurement_value'] >= floor))
    df_q = df_subset[keep].copy()
    df_q['log_value'] = np.log10(df_q['measurement_value'])
    rows = []
    for allele, grp in df_q.groupby('allele'):
        if len(grp) < MIN_N:
            continue
        rho, p = spearmanr(grp[SCORE_COL], grp['log_value'])
        rows.append({
            'Allele': allele,
            'Measurement': measurement_label,
            'n': len(grp),
            'Spearman ρ': round(rho, 3),
            'p-value': p,
            'Significant (p<0.05)': 'yes' if p < 0.05 else 'no',
        })
    cols = ['Allele', 'Measurement', 'n', 'Spearman ρ', 'p-value', 'Significant (p<0.05)']
    return pd.DataFrame(rows, columns=cols)

S3_ic50 = per_allele_spearman(ic50, IC50_CENSORED, IC50_FLOOR, 'IC50')
S3_kd   = per_allele_spearman(kd,   KD_CENSORED,   KD_FLOOR,   'Kd')

print(f'S3_ic50: {len(S3_ic50)} alleles | S3_kd: {len(S3_kd)} alleles')

rows without I_sc_best: 0


IC50 rows: 21,227   Kd rows: 97,743


S3_ic50: 35 alleles | S3_kd: 80 alleles


In [7]:
# Report usable (post-censoring) totals for the manuscript
for label, sub, cens, floor in [('IC50', ic50, IC50_CENSORED, IC50_FLOOR),
                                ('Kd',   kd,   KD_CENSORED,   KD_FLOOR)]:
    keep = ~(sub['measurement_value'].isin(cens) | (sub['measurement_value'] >= floor))
    print(f'{label}: {keep.sum():,} usable of {len(sub):,} total')

IC50: 18,113 usable of 21,227 total
Kd: 48,395 usable of 97,743 total


In [8]:
for label, sub in [('IC50', S3_ic50), ('Kd', S3_kd)]:
    n_sig = (sub['p-value'] < 0.05).sum()
    print(f'{label:>4}: n_alleles={len(sub):>3}, '
          f'median ρ={sub["Spearman ρ"].median():+.3f}, '
          f'IQR [{sub["Spearman ρ"].quantile(0.25):+.2f}, {sub["Spearman ρ"].quantile(0.75):+.2f}], '
          f'{n_sig} significant at p<0.05')

IC50: n_alleles= 35, median ρ=+0.207, IQR [+0.10, +0.37], 20 significant at p<0.05
  Kd: n_alleles= 80, median ρ=+0.172, IQR [+0.05, +0.24], 47 significant at p<0.05


In [9]:
# Final S3 (per-allele Spearman) with formatted p-values for display
S3 = pd.concat([S3_ic50, S3_kd], ignore_index=True)
S3['p-value'] = S3['p-value'].apply(lambda x: '< 0.001' if x < 0.001 else f'{x:.3f}')
S3 = S3.sort_values(['Measurement', 'Allele']).reset_index(drop=True)
S3

,Allele,Measurement,n,Spearman ρ,p-value,Significant (p<0.05)
0,A*01:01,IC50,197,0.306,< 0.001,yes
1,A*02:01,IC50,5319,0.389,< 0.001,yes
2,A*02:02,IC50,2366,0.331,< 0.001,yes
3,A*02:03,IC50,2373,0.295,< 0.001,yes
4,A*02:05,IC50,22,-0.083,0.712,no
...,...,...,...,...,...,...
110,C*07:02,Kd,130,0.125,0.156,no
111,C*08:02,Kd,42,0.348,0.024,yes
112,C*12:03,Kd,161,-0.070,0.380,no
113,C*14:02,Kd,222,0.223,< 0.001,yes


## Table S4: Structural validation pairs

In [10]:
rmsd_df = pd.read_csv(RMSD_CSV)
print('Shape:', rmsd_df.shape)
print('Columns:', list(rmsd_df.columns))

Shape: (76, 51)
Columns: ['allele_iedb', 'allele', 'allele_compact', 'peptide', 'peptide_length', 'measurement_type', 'measurement_value', 'measurement_units', 'assay_method', 'assay_response', 'pubmed_id', 'parent_protein', 'protein_accession', 'source_organism', 'source_version', 'flagged', 'self_templated', 'has_structures', 'num_pdbs', 'I_sc_best', 'I_sc_mean', 'reweighted_sc_best', 'reweighted_sc_mean', 'total_score_best', 'total_score_mean', 'rosetta_best_score', 'rosetta_mean_score', 'pdb_dir', 'pep_sc_best', 'pep_sc_mean', 'matched_pdb_id', 'mhc_chain_id', 'peptide_chain_id', 'resolution_angstrom', 'rmsd_min_of_25', 'n_decoys', 'mhc_alignment_rmsd', 'rmsd_best_score', 'rmsd_top5_mean', 'best_score', 'rmsd_best_score_reweighted_sc', 'rmsd_top5_mean_reweighted_sc', 'best_score_reweighted_sc', 'rmsd_best_score_total_score', 'rmsd_top5_mean_total_score', 'best_score_total_score', 'allele_dir', 'template_pdb', 'template_peptide', 'template_identity', 'rmsd_template']


In [11]:
rename_map = {
    'allele':               'Allele',
    'peptide':               'Peptide',
    'peptide_length':        'Length',
    'matched_pdb_id':        'Experimental PDB',
    'template_pdb':          'Template PDB',
    'template_identity':     'Template identity (%)',
    'rmsd_template':          'Relaxed starting-model RMSD (Å)',
    'rmsd_best_score':       'RMSD best-by-score (Å)',
    'rmsd_top5_mean':        'RMSD top-5 mean (Å)',
    'rmsd_min_of_25':        'RMSD best-overall (Å)',
}
keep_cols = [c for c in rename_map if c in rmsd_df.columns]
S4 = rmsd_df[keep_cols].rename(columns=rename_map).copy()

# Template identity to %: if stored as fraction ≤ 1, multiply by 100.
# These three conditions named S3, not S4, until 2026-09-15. S3 is the Spearman
# table and has no identity column, so every branch was False: the fraction
# conversion never ran and the table never got sorted.
if 'Template identity (%)' in S4.columns and S4['Template identity (%)'].dropna().max() <= 1.5:
    S4['Template identity (%)'] = S4['Template identity (%)'] * 100

# Round numeric columns
for c in ['RMSD best-by-score (Å)', 'RMSD top-5 mean (Å)', 'RMSD best-overall (Å)',
          'RMSD best-overall (Å)', 'Relaxed starting-model RMSD (Å)']:
    if c in S4.columns:
        S4[c] = S4[c].round(2)
if 'Template identity (%)' in S4.columns:
    S4['Template identity (%)'] = S4['Template identity (%)'].round(1)

# Sort by template identity descending (no-template pairs sink to bottom)
if 'Template identity (%)' in S4.columns:
    S4 = S4.sort_values('Template identity (%)', ascending=False, na_position='last').reset_index(drop=True)

print(f'{len(S4)} rows')
S4

76 rows


,Allele,Peptide,Length,Experimental PDB,Template PDB,Template identity (%),Relaxed starting-model RMSD (Å),RMSD best-by-score (Å),RMSD top-5 mean (Å),RMSD best-overall (Å)
0,A*02:01,ALWGPDPAAA,10,3UTQ,5C0D,90.0,0.85,0.79,0.70,0.58
1,B*27:05,KRWIILGLNK,10,4G9D,4G8I,90.0,0.93,1.40,1.21,0.94
2,A*02:01,YLVVVGAVGV,10,6O51,6O53,90.0,0.89,1.61,1.53,1.13
3,A*24:02,RYPLTFGWCF,10,3VXN,5HGD,90.0,1.27,1.26,1.16,0.93
4,A*02:01,EAAGIGILTV,10,2GT9,1JF1,90.0,0.86,0.93,0.89,0.81
5,A*02:01,RLQSLQTYV,9,7N1E,8GON,88.9,0.88,0.87,1.05,0.77
6,A*02:01,KLVVVAVGV,9,6O4Z,6O4Y,88.9,0.51,1.09,1.25,0.90
7,A*02:01,YLQPRTFLL,9,7P3D,7P3E,88.9,1.24,1.27,1.32,1.23
8,A*02:01,SLYNTVATL,9,2V2W,5NMH,88.9,1.13,0.99,1.22,0.91
9,A*01:01,CTELKLSDY,9,4NQV,4NQX,88.9,0.65,0.90,0.75,0.61


## Table S5: Per-allele dataset composition

In [12]:
# S4 describes the full released dataset, INCLUDING the 15 flagged records
# (they are in the release). Correlation tables (S2) exclude them.
df_all = pd.read_csv(METADATA_CSV)
tl = df_all['measurement_type'].astype(str).str.lower()
all_is_ic50 = tl.str.contains('ic50', na=False)
all_is_kd   = tl.str.fullmatch('kd', na=False)

comp_rows = []
for allele, grp in df_all.groupby('allele'):
    n_pep   = grp['peptide'].nunique()
    n_pairs = grp[['allele', 'peptide']].drop_duplicates().shape[0]
    n_ic50  = all_is_ic50.loc[grp.index].sum()
    n_kd    = all_is_kd.loc[grp.index].sum()
    lens    = grp['peptide'].dropna().str.len()
    comp_rows.append({
        'Allele': allele,
        'Unique peptides':   int(n_pep),
        'Peptide-HLA pairs': int(n_pairs),
        'IC50 measurements': int(n_ic50),
        'Kd measurements':   int(n_kd),
        'Length range':      f'{lens.min()}-{lens.max()}' if len(lens) else ', ',
    })

S5 = pd.DataFrame(comp_rows).sort_values('Peptide-HLA pairs', ascending=False).reset_index(drop=True)
S5['% of dataset'] = (S5['Peptide-HLA pairs'] / S5['Peptide-HLA pairs'].sum() * 100).round(2)

pair_total = S5['Peptide-HLA pairs'].sum()
meas_total = S5['IC50 measurements'].sum() + S5['Kd measurements'].sum()
# Cross-check against the file rather than against a number typed in here: a
# hardcoded total silently becomes a tripwire for the next release. These must
# agree with the abstract and Data Records, so print them for the manuscript.
expected_pairs = df_all[['allele', 'peptide']].drop_duplicates().shape[0]
expected_meas  = len(df_all)
print(f'S4 pair total:        {pair_total:,}  (metadata.csv: {expected_pairs:,})')
print(f'S4 measurement total: {meas_total:,}  (metadata.csv: {expected_meas:,})')
assert pair_total == expected_pairs, (
    f'S4 pairs {pair_total:,} != metadata.csv {expected_pairs:,}; the per-allele '
    'grouping is dropping or duplicating pairs')
assert meas_total == expected_meas, (
    f'S4 measurements {meas_total:,} != metadata.csv {expected_meas:,}; a '
    'measurement_type other than IC50/Kd is present and is not being counted')
S5

S4 pair total:        112,561  (metadata.csv: 112,561)
S4 measurement total: 118,985  (metadata.csv: 118,985)


,Allele,Unique peptides,Peptide-HLA pairs,IC50 measurements,Kd measurements,Length range,% of dataset
0,A*02:01,10376,10376,6261,4252,7-15,9.22
1,A*03:01,5893,5893,591,5312,8-12,5.24
2,A*11:01,5191,5191,667,4529,8-11,4.61
3,A*31:01,4572,4572,141,4431,8-11,4.06
4,A*68:02,4233,4233,2451,1782,8-11,3.76
5,A*02:03,4078,4078,2457,3681,8-11,3.62
6,A*01:01,3534,3534,306,3252,8-12,3.14
7,A*02:06,3518,3518,2457,3118,8-11,3.13
8,B*15:01,3456,3456,283,3189,8-14,3.07
9,B*07:02,3346,3346,563,2807,8-12,2.97


In [13]:
S6 = pd.read_csv(SCORES_DIR / 'metric_comparison_pooled.csv')
S6 = S6.rename(columns={'assay':'Assay', 'metric':'Score', 'agg':'Metric',
                        'rho':'Spearman ρ'})
# Two decimals, rounded once from the raw value. Rounding to 3 dp here and
# letting a reader round again to 2 turns 0.31467 into 0.32; the article
# reports correlations to two decimals, so do it in one step.
S6['Spearman ρ'] = S6['Spearman ρ'].round(2)
S6['p'] = S6['p'].map(lambda x: f'{x:.1e}')
print(S6.to_string(index=False))

Assay         Score Metric  Spearman ρ        p     n
 IC50          I_sc   best        0.31  0.0e+00 18113
 IC50          I_sc   mean        0.24 7.1e-236 18113
 IC50        pep_sc   best        0.14  1.7e-79 18113
 IC50        pep_sc   mean        0.04  6.8e-09 18113
 IC50 reweighted_sc   best        0.13  2.8e-65 18113
 IC50 reweighted_sc   mean        0.03  7.6e-06 18113
 IC50   total_score   best        0.01  6.3e-02 18113
 IC50   total_score   mean       -0.02  9.2e-04 18113
   Kd          I_sc   best        0.18  0.0e+00 48395
   Kd          I_sc   mean        0.16 1.3e-265 48395
   Kd        pep_sc   best        0.11 6.5e-140 48395
   Kd        pep_sc   mean        0.08  6.6e-69 48395
   Kd reweighted_sc   best        0.12 2.6e-151 48395
   Kd reweighted_sc   mean        0.07  2.1e-52 48395
   Kd   total_score   best        0.06  1.6e-35 48395
   Kd   total_score   mean        0.03  2.2e-13 48395


## Save all tables to one .xlsx

One sheet per table, open in Excel, copy individual tables into Word.

In [14]:
out_xlsx = OUT_DIR / 'PepBind3D_supplemental_tables.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as xw:
    S1.to_excel(xw, sheet_name='S1_curation_logic',      index=False)
    S2.to_excel(xw, sheet_name='S2_attrition',           index=False)
    S3.to_excel(xw, sheet_name='S3_per_allele_spearman', index=False)
    S4.to_excel(xw, sheet_name='S4_validation_pairs',    index=False)
    S5.to_excel(xw, sheet_name='S5_allele_composition',  index=False)
    S6.to_excel(xw, sheet_name='S6_metric_comparison',   index=False)

print(f'Wrote: {out_xlsx}')
print('Sheets: S1-S6')

Wrote: <DATA_ROOT>/IEDB_validation/supplemental_tables/PepBind3D_supplemental_tables.xlsx
Sheets: S1-S6


In [15]:
# Copy the workbook into revisions/tables/ for the draft.
#
# One .xlsx with six sheets, not six CSVs: the journal takes oversize
# supplementary tables as xlsx, and a parallel set of CSVs is a second copy of
# the same numbers that goes stale the first time only one of them is rebuilt.
import shutil
from paths import REVISIONS_DIR, REPO_ROOT

TABLES = {
    'S1_curation_logic':      S1,
    'S2_attrition':           S2,
    'S3_per_allele_spearman': S3,
    'S4_validation_pairs':    S4,
    'S5_allele_composition':  S5,
    'S6_metric_comparison':   S6,
}

REV_TABLES = REVISIONS_DIR / 'tables'
REV_TABLES.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_xlsx, REV_TABLES / out_xlsx.name)

# Clear out the CSVs an earlier version of this notebook wrote, so the
# directory does not keep serving a stale second copy.
for stale in list(OUT_DIR.glob('*.csv')) + list(REV_TABLES.glob('*.csv')):
    stale.unlink()

# Print roots relative to the repo, never absolute: this notebook's stored
# output is committed, and the repository is public.
print(f'copied {out_xlsx.name} -> {REV_TABLES.relative_to(REPO_ROOT)}/')
for name, tbl in TABLES.items():
    print(f'  {name:<26} {tbl.shape[0]:>4} rows x {tbl.shape[1]} cols')


copied PepBind3D_supplemental_tables.xlsx -> revisions/tables/
  S1_curation_logic            13 rows x 4 cols
  S2_attrition                  6 rows x 4 cols
  S3_per_allele_spearman      115 rows x 6 cols
  S4_validation_pairs          76 rows x 10 cols
  S5_allele_composition        95 rows x 7 cols
  S6_metric_comparison         16 rows x 6 cols
